<a href="https://colab.research.google.com/github/manavkashyap2453-cell/Bank-Fraud-Detection/blob/main/sql_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Initial Setup**

In [4]:
import pandas as pd
import sqlite3
import warnings
import os

warnings.filterwarnings('ignore')

In [5]:
path='/content/Bank_Transaction_Fraud_Detection.csv'
flag=os.path.exists(path)

try:
  df_explore=pd.read_csv(path)
except:
  print("Dataset not found")

In [7]:
conn = sqlite3.connect(':memory:')

# dataframe to the SQL database as a table named 'transactions'
df_explore.to_sql('transactions', conn, index=False, if_exists='replace')

def run_query(query):
    return pd.read_sql_query(query, conn)

In [9]:
df_explore.columns

Index(['Customer_ID', 'Customer_Name', 'Gender', 'Age', 'State', 'City',
       'Bank_Branch', 'Account_Type', 'Transaction_ID', 'Transaction_Date',
       'Transaction_Time', 'Transaction_Amount', 'Merchant_ID',
       'Transaction_Type', 'Merchant_Category', 'Account_Balance',
       'Transaction_Device', 'Transaction_Location', 'Device_Type', 'Is_Fraud',
       'Transaction_Currency', 'Customer_Contact', 'Transaction_Description',
       'Customer_Email'],
      dtype='object')

# **Sql analysis**

**1. High-Risk Multi-Category Combinations**

Identifies which specific combinations of account types, merchant segments, and generalized device tiers suffer from the worst fraud ratios.

In [12]:
query1="""
SELECT
    Account_Type,
    Merchant_Category,
    Device_Type,
    COUNT(*) AS Total_Tx,
    SUM(Is_Fraud) AS Fraud_Count,
    ROUND(AVG(Is_Fraud) * 100, 2) AS Fraud_Rate_Pct
FROM transactions
GROUP BY Account_Type, Merchant_Category, Device_Type
HAVING Fraud_Count > 0
ORDER BY Fraud_Rate_Pct DESC, Total_Tx DESC
LIMIT 10;
"""

pd.read_sql_query(query1,conn)

,Account_Type,Merchant_Category,Device_Type,Total_Tx,Fraud_Count,Fraud_Rate_Pct
0,Business,Groceries,POS,2691,171,6.35
1,Business,Health,Desktop,2745,163,5.94
2,Checking,Electronics,ATM,2782,161,5.79
3,Business,Clothing,POS,2750,158,5.75
4,Savings,Restaurant,ATM,2746,158,5.75
5,Business,Restaurant,Desktop,2719,155,5.70
6,Savings,Groceries,POS,2804,158,5.63
7,Business,Entertainment,ATM,2770,156,5.63
8,Business,Groceries,Desktop,2766,154,5.57
9,Savings,Groceries,Mobile,2736,152,5.56


**2. The Mismapped Device Anomaly**

In [13]:
query2="""
SELECT
    Device_Type,
    Transaction_Device,
    COUNT(*) AS Total_Tx,
    SUM(Is_Fraud) AS Fraud_Count,
    ROUND(AVG(Is_Fraud) * 100, 2) AS Fraud_Rate_Pct
FROM transactions
GROUP BY Device_Type, Transaction_Device
ORDER BY Fraud_Rate_Pct DESC;
"""

pd.read_sql_query(query2,conn)

,Device_Type,Transaction_Device,Total_Tx,Fraud_Count,Fraud_Rate_Pct
0,Desktop,Voice Assistant,1994,129,6.47
1,Desktop,Payment Gateway Device,1986,124,6.24
2,Mobile,Virtual Card,1950,116,5.95
3,ATM,Web Browser,2007,119,5.93
4,ATM,Biometric Scanner,2005,117,5.84
...,...,...,...,...,...
75,Mobile,Wearable Device,2089,92,4.40
76,Mobile,Voice Assistant,1933,84,4.35
77,Mobile,POS Mobile App,1937,84,4.34
78,Desktop,Banking Chatbot,1999,79,3.95


**3. State-Level Geographic Risk Matrix**

In [15]:
query3="""
SELECT
    State,
    COUNT(*) AS Total_Tx,
    SUM(Is_Fraud) AS Total_Fraud_Tx,
    ROUND(SUM(Is_Fraud) * 100.0 / COUNT(*), 2) AS State_Fraud_Rate_Pct
FROM transactions
GROUP BY State
ORDER BY State_Fraud_Rate_Pct DESC;
"""

pd.read_sql_query(query3,conn)

,State,Total_Tx,Total_Fraud_Tx,State_Fraud_Rate_Pct
0,Tamil Nadu,5841,328,5.62
1,Lakshadweep,5954,332,5.58
2,Sikkim,5793,312,5.39
3,Uttarakhand,5985,319,5.33
4,West Bengal,5851,311,5.32
5,Madhya Pradesh,5928,315,5.31
6,Mizoram,5892,312,5.30
7,Telangana,5952,315,5.29
8,Tripura,5869,308,5.25
9,Karnataka,5775,303,5.25


**4. Gender Bias vs. Merchant Segment Fraud Metrics**

In [26]:
query4="""
SELECT
    Gender,
    Merchant_Category,
    COUNT(*) AS Total_Transactions,
    SUM(Is_Fraud) AS Fraud_Transactions,
    ROUND(AVG(Is_Fraud) * 100, 2) AS Group_Fraud_Rate_Pct
FROM transactions
GROUP BY Gender, Merchant_Category
ORDER BY Group_Fraud_Rate_Pct DESC;
"""

pd.read_sql_query(query4,conn)

,Gender,Merchant_Category,Total_Transactions,Fraud_Transactions,Group_Fraud_Rate_Pct
0,Female,Clothing,16588,864,5.21
1,Male,Groceries,16683,868,5.20
2,Male,Clothing,16752,870,5.19
3,Female,Groceries,16504,854,5.17
4,Female,Health,16472,835,5.07
5,Female,Restaurant,16622,841,5.06
6,Male,Electronics,16618,838,5.04
7,Female,Electronics,16791,843,5.02
8,Male,Restaurant,16903,847,5.01
9,Male,Entertainment,16850,842,5.00


**5. Low-Variance Feature Check Validation**

In [18]:
query5="""
SELECT
    Transaction_Currency,
    COUNT(*) AS Total_Tx,
    SUM(Is_Fraud) AS Fraud_Count,
    AVG(Is_Fraud) AS Baseline_Check
FROM transactions
GROUP BY Transaction_Currency;
"""

pd.read_sql_query(query5,conn)

,Transaction_Currency,Total_Tx,Fraud_Count,Baseline_Check
0,INR,200000,10088,0.05044


**6.Disproportionate value exhaution**

In [19]:
query6="""
SELECT
    Is_Fraud,
    COUNT(*) AS Tx_Count,
    ROUND(AVG(Transaction_Amount), 2) AS Avg_Spent,
    ROUND(AVG(Account_Balance), 2) AS Avg_Remaining_Balance
FROM transactions
WHERE Transaction_Amount > Account_Balance
GROUP BY Is_Fraud;
"""

pd.read_sql_query(query6,conn)

,Is_Fraud,Tx_Count,Avg_Spent,Avg_Remaining_Balance
0,0,89366,67666.39,36266.41
1,1,4705,67975.51,36706.32


**7. Age group bracket risk segmentation**

In [21]:
query7="""
SELECT
    CASE
        WHEN Age BETWEEN 18 AND 25 THEN '18-25 (Young Adult)'
        WHEN Age BETWEEN 26 AND 40 THEN '26-40 (Prime)'
        WHEN Age BETWEEN 41 AND 60 THEN '41-60 (Middle Age)'
        ELSE '61-70 (Senior)'
    END AS Age_Group,
    COUNT(*) AS Total_Tx,
    SUM(Is_Fraud) AS Fraud_Count,
    ROUND(AVG(Is_Fraud) * 100, 2) AS Fraud_Rate_Pct
FROM transactions
GROUP BY Age_Group
ORDER BY Fraud_Rate_Pct DESC;
"""

pd.read_sql_query(query7,conn)

,Age_Group,Total_Tx,Fraud_Count,Fraud_Rate_Pct
0,18-25 (Young Adult),30021,1574,5.24
1,41-60 (Middle Age),75289,3794,5.04
2,26-40 (Prime),56837,2836,4.99
3,61-70 (Senior),37853,1884,4.98


**8. Extreme deviation form national averages**

In [22]:
query8="""
SELECT
    State,
    City,
    ROUND(AVG(Transaction_Amount), 2) AS City_Avg_Spent,
    ROUND(AVG(Transaction_Amount) - (SELECT AVG(Transaction_Amount) FROM transactions), 2) AS Deviation_From_Global_Avg
FROM transactions
GROUP BY State, City
HAVING COUNT(*) > 500
ORDER BY City_Avg_Spent DESC
LIMIT 10;
"""

pd.read_sql_query(query8,conn)

,State,City,City_Avg_Spent,Deviation_From_Global_Avg
0,Kerala,Kochi,51716.20,2178.18
1,Meghalaya,Tura,51454.68,1916.67
2,Telangana,Karimnagar,51397.16,1859.14
3,West Bengal,Siliguri,51352.53,1814.51
4,Bihar,Munger,51283.23,1745.21
5,Uttarakhand,Dehradun,50977.91,1439.89
6,Jharkhand,Hazaribagh,50945.25,1407.23
7,Meghalaya,Jowai,50843.72,1305.70
8,Himachal Pradesh,Kullu,50803.92,1265.90
9,Bihar,Bhagalpur,50783.19,1245.17


**9. Account type balance: Liquidity vs Fraud risk**

In [23]:
query9="""
SELECT
    Account_Type,
    CASE
        WHEN Account_Balance < 25000 THEN 'Low Balance (<25k)'
        WHEN Account_Balance BETWEEN 25000 AND 75000 THEN 'Medium Balance (25k-75k)'
        ELSE 'High Balance (>75k)'
    END AS Balance_Tier,
    COUNT(*) AS Total_Tx,
    SUM(Is_Fraud) AS Fraud_Count,
    ROUND(AVG(Is_Fraud) * 100, 2) AS Fraud_Rate_Pct
FROM transactions
GROUP BY Account_Type, Balance_Tier
ORDER BY Account_Type, Fraud_Rate_Pct DESC;
"""

pd.read_sql_query(query9,conn)

,Account_Type,Balance_Tier,Total_Tx,Fraud_Count,Fraud_Rate_Pct
0,Business,High Balance (>75k),17441,930,5.33
1,Business,Medium Balance (25k-75k),35123,1816,5.17
2,Business,Low Balance (<25k),13919,690,4.96
3,Checking,Medium Balance (25k-75k),35250,1795,5.09
4,Checking,Low Balance (<25k),14120,708,5.01
5,Checking,High Balance (>75k),17554,800,4.56
6,Savings,High Balance (>75k),17405,911,5.23
7,Savings,Medium Balance (25k-75k),35128,1742,4.96
8,Savings,Low Balance (<25k),14060,696,4.95


**10.Zero balance exposure flag**

In [25]:
query10="""
SELECT
    State, City, Account_Type, Transaction_Amount, Account_Balance, Is_Fraud
FROM transactions
WHERE Account_Balance < 10000 AND Transaction_Amount > 80000
ORDER BY Transaction_Amount DESC
LIMIT 10;
"""

pd.read_sql_query(query10,conn)

,State,City,Account_Type,Transaction_Amount,Account_Balance,Is_Fraud
0,Chandigarh,Chandigarh,Checking,98980.22,6025.73,0
1,Jharkhand,Dhanbad,Savings,98974.37,6390.11,0
2,Mizoram,Champhai,Business,98971.74,6116.14,0
3,Sikkim,Gangtok,Checking,98970.43,7916.24,0
4,Meghalaya,Tura,Checking,98939.23,9227.09,0
5,Assam,Dibrugarh,Business,98938.71,5836.55,0
6,Kerala,Trichur,Checking,98917.72,7163.86,0
7,Haryana,Gurugram,Business,98902.75,8626.44,0
8,Madhya Pradesh,Ujjain,Checking,98897.91,8485.25,0
9,Chhattisgarh,Korba,Business,98893.58,7215.97,0



## Quick Insights Summary

* **Top Risk Vectors:** Business accounts shopping for **Groceries via POS** ($6.35\%$) and **Health via Desktop** ($5.94\%$) show the highest fraud rates.
* **ATM Vulnerabilities:** Savings and Checking accounts used at ATMs for **Electronics** ($5.79\%$) and **Restaurants** ($5.75\%$) are primary fraud targets.
* **Technical Anomalies:** High fraud rates on mismatched hardware (e.g., **Desktops** on *Voice Assistants* at $6.47\%$ or **ATMs** on *Web Browsers* at $5.93\%$) signal **API spoofing** and bot emulation.
* **Geographic & Demographic Tiers:** **Tamil Nadu** ($5.62\%$) and **Lakshadweep** ($5.58\%$) are the top risk states. Age-wise, young adults (**18–25**) carry the highest baseline risk ($5.24\%$).
* **Gender Key Segments:** **Female/Clothing** ($5.21\%$) and **Male/Groceries** ($5.20\%$) are the most targeted gender-merchant intersections.
* **Behavioral Reality:** Large transactions (over ₹80,000) that completely drain account balances (under ₹10,000) frequently belong to legitimate users (`Is_Fraud = 0`); balance depletion alone is a poor fraud indicator.
* **Baseline Control:** The global transaction baseline fraud rate sits uniformly at **$5.04\%$** (all in **INR**).

---